# RAG Colab Notebook
This single notebook runs in Google Colab. It installs dependencies and provides an interactive `ipywidgets` UI to upload a PDF, choose a domain (Media, Law, Telecom, General), and either ask questions or generate a structured summary using a Retrieval-Augmented Generation (RAG) pipeline.

Run the first code cell to install dependencies and set your `OPENAI_API_KEY` value (in Cell 1). Then run the second cell to show the UI.

In [ ]:
# Colab Cell 1: Install dependencies
# Run this cell first. Do NOT pin langchain to 0.3.x — Colab pre-installs
# langchain-classic / langgraph which require langchain-core >= 1.4.4.

!pip install -q \
  langchain \
  langchain-openai \
  langchain-community \
  langchain-chroma \
  langchain-text-splitters \
  chromadb \
  pypdf \
  ipywidgets \
  sentence-transformers

# Enable ipywidgets rendering in Colab
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass

# Print installed versions for verification
try:
    import importlib.metadata as _meta
except ImportError:
    import importlib_metadata as _meta

for _pkg in [
    "langchain", "langchain-core", "langchain-openai",
    "langchain-chroma", "langchain-text-splitters",
    "chromadb", "pypdf", "ipywidgets", "sentence-transformers",
]:
    try:
        print(f"{_pkg}=={_meta.version(_pkg)}")
    except Exception:
        print(f"{_pkg} not found")

print("\nInstallation complete. Run Cell 2 to launch the UI.")
print("You will enter your OpenAI API key directly in the UI.")

In [ ]:
# Colab Cell 2: RAG app with ipywidgets UI
# Run after Cell 1. Enter your OpenAI API key in the UI field below.

import os
import io
import threading
from pypdf import PdfReader
from IPython.display import display, Markdown
import ipywidgets as widgets
from uuid import uuid4

# Re-enable widget manager in case Cell 1 wasn't re-run
try:
    from google.colab import output as _colab_out
    _colab_out.enable_custom_widget_manager()
except Exception:
    pass

# Modern LangChain import paths (compatible with langchain-core >= 1.4.4)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── Utility ───────────────────────────────────────────────────────────────────

def pdf_bytes_to_text(pdf_bytes):
    reader = PdfReader(io.BytesIO(pdf_bytes))
    pages = []
    for p in reader.pages:
        try:
            pages.append(p.extract_text() or "")
        except Exception:
            pages.append("")
    return "\n\n".join(pages)

def get_api_key():
    return api_key_input.value.strip()

def build_vectorstore_with_files(files, existing_vectordb=None, persist_dir=None):
    """Index list of (fname, bytes) into Chroma. Adds to existing_vectordb if given."""
    openai_key = get_api_key()
    embedding_fn = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_key=openai_key)
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

    all_texts, all_metadatas = [], []
    for fname, pdf_bytes in files:
        text = pdf_bytes_to_text(pdf_bytes)
        chunks = splitter.split_text(text)
        all_texts.extend(chunks)
        all_metadatas.extend([{"source": fname}] * len(chunks))

    if existing_vectordb is None:
        persist_dir = persist_dir or f"chroma_store_{uuid4().hex}"
        vectordb = Chroma.from_texts(
            texts=all_texts,
            embedding=embedding_fn,
            metadatas=all_metadatas,
            persist_directory=persist_dir,
        )
    else:
        existing_vectordb.add_texts(texts=all_texts, metadatas=all_metadatas)
        vectordb = existing_vectordb

    return vectordb.as_retriever(search_kwargs={"k": 5}), vectordb, persist_dir

def get_llm():
    return ChatOpenAI(model="gpt-4o-mini", openai_api_key=get_api_key(), temperature=0.0)

# ── Prompts ───────────────────────────────────────────────────────────────────

BASE_PROMPT = """You are a helpful assistant that strictly answers only from the provided CONTEXT.
Domain: {domain}

RULES:
- Use only the provided CONTEXT to answer.
- If the answer cannot be found, respond exactly: "I cannot find that information in the provided document."
- Keep answers concise and focused.
- When the domain is "Law", emphasize definitions, clauses, and liabilities.
- When the domain is "Telecom" or "Media", emphasize technical specs, service terms, or metrics.
- When the domain is "General", provide concise, neutral answers.
- For any other domain, apply relevant expertise appropriate to that field.

CONTEXT:
{context}

QUESTION: {question}

Answer:"""

SUMMARY_PROMPT = """You are a summarization assistant. Use only the CONTEXT below.
Domain: {domain}

If important details are missing, state "I cannot find that information in the provided document."

CONTEXT:
{context}

Produce a concise structured summary with headings where useful."""

QA_TEMPLATE = PromptTemplate(template=BASE_PROMPT, input_variables=["context", "question", "domain"])
SUMMARY_TEMPLATE = PromptTemplate(template=SUMMARY_PROMPT, input_variables=["context", "domain"])

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# ── RAG functions ─────────────────────────────────────────────────────────────

def query_with_rag(retriever, question, domain):
    llm = get_llm()
    chain = (
        {
            "context": retriever | format_docs,
            "question": RunnablePassthrough(),
            "domain": lambda _: domain,
        }
        | QA_TEMPLATE
        | llm
        | StrOutputParser()
    )
    result = chain.invoke(question)
    if "I cannot find that information in the provided document." in result:
        return "I cannot find that information in the provided document."
    return result

def generate_summary(retriever, domain):
    llm = get_llm()
    docs = retriever.invoke("summary overview")
    combined = "\n\n".join(d.page_content for d in docs)
    prompt_text = SUMMARY_TEMPLATE.format(context=combined, domain=domain)
    result = llm.invoke(prompt_text)
    text = result.content if hasattr(result, "content") else str(result)
    if "I cannot find that information in the provided document." in text:
        return "I cannot find that information in the provided document."
    return text

# ── State ─────────────────────────────────────────────────────────────────────

_state = {
    "vectordb": None,
    "retriever": None,
    "persist_dir": None,
    "indexed_files": [],
}

# ── Widgets ───────────────────────────────────────────────────────────────────

# Step 1 — API key
api_key_input = widgets.Password(
    value="",
    placeholder="sk-...",
    description="OpenAI Key:",
    layout=widgets.Layout(width="400px"),
)
api_key_status = widgets.HTML('<span style="color:orange">Enter your OpenAI API key</span>')

def _on_key_change(change):
    key = change["new"].strip()
    if key.startswith("sk-") and len(key) > 20:
        api_key_status.value = '<span style="color:green">&#10003; Key set</span>'
    else:
        api_key_status.value = '<span style="color:orange">Enter a valid sk-... key</span>'

api_key_input.observe(_on_key_change, names="value")

# Step 2 — Category selector with custom category support
_DEFAULT_DOMAINS = ["Media", "Law", "Telecom", "General"]

domain_dropdown = widgets.Dropdown(
    options=_DEFAULT_DOMAINS,
    value="General",
    description="Category:",
    layout=widgets.Layout(width="220px"),
)
new_category_input = widgets.Text(
    value="",
    placeholder="e.g. Finance, Healthcare…",
    description="",
    layout=widgets.Layout(width="210px"),
)
add_category_btn = widgets.Button(description="+ Add", button_style="warning", layout=widgets.Layout(width="70px"))
remove_category_btn = widgets.Button(description="✕ Remove", button_style="danger", layout=widgets.Layout(width="90px"))
category_msg = widgets.HTML("")

def on_add_category(b):
    name = new_category_input.value.strip()
    if not name:
        category_msg.value = '<span style="color:orange">Type a category name first.</span>'
        return
    opts = list(domain_dropdown.options)
    if name in opts:
        category_msg.value = f'<span style="color:orange">"{name}" already exists.</span>'
        return
    opts.append(name)
    domain_dropdown.options = opts
    domain_dropdown.value = name
    new_category_input.value = ""
    category_msg.value = f'<span style="color:green">&#10003; Added "{name}"</span>'

add_category_btn.on_click(on_add_category)

def on_remove_category(b):
    current = domain_dropdown.value
    if current in _DEFAULT_DOMAINS:
        category_msg.value = f'<span style="color:orange">Cannot remove built-in category "{current}".</span>'
        return
    opts = [o for o in domain_dropdown.options if o != current]
    domain_dropdown.options = opts
    domain_dropdown.value = opts[-1] if opts else None
    category_msg.value = f'<span style="color:green">&#10003; Removed "{current}"</span>'

remove_category_btn.on_click(on_remove_category)

# Step 2 — Upload via google.colab.files.upload() (reliable in Colab)
# The Output widget captures the native file-picker UI inline.
upload_out = widgets.Output(layout=widgets.Layout(
    border="1px dashed #555",
    padding="6px",
    min_height="32px",
    margin="4px 0",
))
upload_btn = widgets.Button(
    description="📂 Upload PDF(s)",
    button_style="info",
    layout=widgets.Layout(width="160px"),
    tooltip="Click to open file picker — select one or more PDFs",
)
clear_btn = widgets.Button(
    description="Clear Index",
    button_style="danger",
    layout=widgets.Layout(width="110px"),
    tooltip="Wipe all indexed documents and start fresh",
)
indexed_files_display = widgets.HTML('<i style="color:#888">No documents indexed yet</i>')

def _refresh_indexed_display():
    files = _state["indexed_files"]
    if not files:
        indexed_files_display.value = '<i style="color:#888">No documents indexed yet</i>'
    else:
        rows = "".join(
            f'<li style="color:green;margin:2px 0">&#10003; {f}</li>'
            for f in files
        )
        indexed_files_display.value = (
            f'<b>Indexed ({len(files)} doc{"s" if len(files) != 1 else ""}):</b>'
            f'<ul style="margin:4px 0;padding-left:18px">{rows}</ul>'
        )

def _index_files_async(pdf_files):
    """Run indexing in a background thread."""
    names = [fn for fn, _ in pdf_files]
    with status_out:
        status_out.clear_output()
        print(f"Indexing {len(pdf_files)} file(s): {', '.join(names)}…")

    def worker():
        try:
            retriever, vectordb, persist_dir = build_vectorstore_with_files(
                pdf_files,
                existing_vectordb=_state["vectordb"],
                persist_dir=_state["persist_dir"],
            )
            _state["vectordb"] = vectordb
            _state["retriever"] = retriever
            _state["persist_dir"] = persist_dir
            for fn in names:
                if fn not in _state["indexed_files"]:
                    _state["indexed_files"].append(fn)
            _refresh_indexed_display()
            with status_out:
                status_out.clear_output()
                print(f"✓ Done. {len(_state['indexed_files'])} doc(s) ready to query.")
        except Exception as e:
            with status_out:
                status_out.clear_output()
                print("Indexing failed:", e)

    threading.Thread(target=worker).start()

def on_upload_btn_click(b):
    if not get_api_key():
        upload_out.clear_output()
        with upload_out:
            print("Please enter your OpenAI API key first.")
        return

    upload_out.clear_output()

    # Use google.colab.files.upload() — this is the reliable upload path in Colab.
    # The file-picker UI renders inside the upload_out Output widget below.
    try:
        from google.colab import files as _gfiles
    except ImportError:
        with upload_out:
            print("Not running in Colab. Cannot open file picker.")
        return

    with upload_out:
        print("Opening file picker — select one or more PDFs…")
        try:
            raw = _gfiles.upload()
        except Exception as e:
            print("Upload error:", e)
            return

    if not raw:
        with upload_out:
            upload_out.clear_output()
            print("No files uploaded.")
        return

    pdf_files = [(fn, bytes(content)) for fn, content in raw.items() if fn.lower().endswith(".pdf")]
    skipped = [fn for fn in raw if not fn.lower().endswith(".pdf")]

    if skipped:
        with upload_out:
            upload_out.clear_output()
            print(f"Skipped non-PDF file(s): {', '.join(skipped)}")
    if not pdf_files:
        with upload_out:
            upload_out.clear_output()
            print("No valid PDF files found — please upload .pdf files.")
        return

    upload_out.clear_output()
    _index_files_async(pdf_files)

upload_btn.on_click(on_upload_btn_click)

def on_clear_btn_click(b):
    _state["vectordb"] = None
    _state["retriever"] = None
    _state["persist_dir"] = None
    _state["indexed_files"] = []
    _refresh_indexed_display()
    with status_out:
        status_out.clear_output()
        print("Index cleared. Upload new PDF(s) to start again.")

clear_btn.on_click(on_clear_btn_click)

# Step 3 — Actions
ask_text = widgets.Text(value="", description="Question:", layout=widgets.Layout(width="70%"))
ask_button = widgets.Button(description="Ask", button_style="primary")
summary_button = widgets.Button(description="Generate Summary", button_style="info")

status_out = widgets.Output(layout=widgets.Layout(border="1px solid #ccc", padding="10px", min_height="40px"))
console_out = widgets.Output(layout=widgets.Layout(border="1px solid #444", padding="10px", min_height="40px"))

# ── Action handlers ───────────────────────────────────────────────────────────

def display_answer(text):
    console_out.clear_output()
    with console_out:
        display(Markdown(text))

def on_ask_clicked(b):
    if not get_api_key():
        with status_out:
            status_out.clear_output()
            print("Please enter your OpenAI API key first.")
        return
    if _state["retriever"] is None:
        with status_out:
            status_out.clear_output()
            print("No documents indexed. Click '📂 Upload PDF(s)' and upload a PDF first.")
        return
    question = ask_text.value.strip()
    if not question:
        with status_out:
            status_out.clear_output()
            print("Please type a question.")
        return
    domain = domain_dropdown.value
    with status_out:
        status_out.clear_output()
        print(f"Processing question across {len(_state['indexed_files'])} doc(s) (domain: {domain})…")
    def worker():
        try:
            ans = query_with_rag(_state["retriever"], question, domain)
            display_answer(ans)
            with status_out:
                status_out.clear_output()
                print("Done.")
        except Exception as e:
            with status_out:
                status_out.clear_output()
                print("Query failed:", e)
    threading.Thread(target=worker).start()

ask_button.on_click(on_ask_clicked)

def on_summary_clicked(b):
    if not get_api_key():
        with status_out:
            status_out.clear_output()
            print("Please enter your OpenAI API key first.")
        return
    if _state["retriever"] is None:
        with status_out:
            status_out.clear_output()
            print("No documents indexed. Click '📂 Upload PDF(s)' and upload a PDF first.")
        return
    domain = domain_dropdown.value
    with status_out:
        status_out.clear_output()
        print(f"Generating summary across {len(_state['indexed_files'])} doc(s) (domain: {domain})…")
    def worker():
        try:
            summ = generate_summary(_state["retriever"], domain)
            display_answer(summ)
            with status_out:
                status_out.clear_output()
                print("Summary generated.")
        except Exception as e:
            with status_out:
                status_out.clear_output()
                print("Summary failed:", e)
    threading.Thread(target=worker).start()

summary_button.on_click(on_summary_clicked)

# ── Layout ────────────────────────────────────────────────────────────────────

key_box = widgets.VBox([
    widgets.HTML("<b>Step 1 — Enter your OpenAI API Key</b>"),
    widgets.HBox([api_key_input, api_key_status]),
])

upload_box = widgets.VBox([
    widgets.HTML("<b>Step 2 — Choose Category &amp; Upload PDF(s)</b>"),
    domain_dropdown,
    widgets.HBox([new_category_input, add_category_btn, remove_category_btn]),
    category_msg,
    widgets.HTML("<div style='margin-top:6px'></div>"),
    widgets.HBox([upload_btn, clear_btn]),
    upload_out,
    indexed_files_display,
])

action_box = widgets.VBox([
    widgets.HTML("<b>Step 3 — Ask or Summarize</b>"),
    widgets.HBox([ask_text, ask_button]),
    widgets.HTML("<i>— or —</i>"),
    summary_button,
])

ui = widgets.VBox([
    key_box,
    widgets.HTML("<hr>"),
    widgets.HBox([upload_box, action_box]),
    widgets.HTML("<hr>"),
    widgets.HTML("<b>Status</b>"),
    status_out,
    widgets.HTML("<b>Answer</b>"),
    console_out,
])

display(ui)